# Solutions to project: counting words with classes

The same moves as the Higgs project, on a book instead of collisions: *Alice's Adventures in Wonderland*, chapter by chapter.
A `Tally` class keeps word counts and knows how to add, and a `WordCounter` class turns each chapter into a result. Results combine key by key, so the whole book is just the sum of its chapters.

**How to work.** Replace every `...` with your own code. When a part is done, or you are properly stuck after five minutes, compare with the matching `solutions-` notebook.

In [ ]:
import json
import re

with open("../data/alice_chapters.json") as f:
    chapters = json.load(f)

[chapter["title"] for chapter in chapters]

`words` turns text into a list of lowercase words. It is given, so that the project is about classes, not about regular expressions.

In [ ]:
def words(text):
    return re.findall(r"[a-z’']+", text.lower())


words("Oh dear! Oh dear! I shall be late!")

## Part 1: `__init__`

`__init__` stores the data on a new object.

The cell below holds the whole class, with three gaps. Complete one method at a time, run the class cell again, then run the part's cell below.

**Task.** Complete `__init__` so that a `Tally` stores its dictionary of counts.

In [ ]:
class Tally:
    def __init__(self, counts):
        self.counts = counts

    def __add__(self, other):
        return ...

    @property
    def total(self):
        return ...

In [ ]:
t = Tally({"alice": 2, "rabbit": 1})
t

## Part 2: `__add__`

`__add__` makes `+` work.

**Task.** Complete `__add__`: it returns a new `Tally` in which each word's count is the sum of its counts in the two. A word missing from one side counts as zero there.
`merged.get(word, 0)` reads a count that may not exist yet.

In [ ]:
class Tally:
    def __init__(self, counts):
        self.counts = counts

    def __add__(self, other):
        merged = dict(self.counts)
        for word, count in other.counts.items():
            merged[word] = merged.get(word, 0) + count
        return Tally(merged)

    @property
    def total(self):
        return ...

In [ ]:
combined = Tally({"alice": 2, "rabbit": 1}) + Tally({"alice": 1, "queen": 4})
combined

## Part 3: A property

A property is computed when you read it.

**Task.** Complete `total`: the sum of all the counts. No brackets when you use it: `t.total`.

In [ ]:
class Tally:
    def __init__(self, counts):
        self.counts = counts

    def __add__(self, other):
        merged = dict(self.counts)
        for word, count in other.counts.items():
            merged[word] = merged.get(word, 0) + count
        return Tally(merged)

    @property
    def total(self):
        return sum(self.counts.values())

In [ ]:
combined = Tally({"alice": 2, "rabbit": 1}) + Tally({"alice": 1, "queen": 4})
combined.total

## Part 4: From text to `Tally`

One function turns a chapter into a `Tally`.

**Task.** Complete `tally`: count how often each word appears in `text`. Loop over `words(text)` and build the dictionary with `counts.get(word, 0) + 1`.

In [ ]:
def tally(text):
    counts = {}
    for word in words(text):
        counts[word] = counts.get(word, 0) + 1
    return Tally(counts)


first = tally(chapters[0]["text"])
first.total

## Part 5: One class runs it all

A class holds the settings and the work.

**Task.** Complete `count`: a `Tally` of the words in `text` that are at least `self.min_length` letters long, and a chapter count of 1.
Build it from `tally(text)` and keep only the long enough words.

In [ ]:
class WordCounter:
    def __init__(self, min_length):
        self.min_length = min_length

    def count(self, text):
        full = tally(text)
        kept = {word: count for word, count in full.counts.items() if len(word) >= self.min_length}
        return {"words": Tally(kept), "chapters": 1}


counter = WordCounter(min_length=5)
output = counter.count(chapters[0]["text"])
output["words"].total

## Part 6: Chapter by chapter

Results from separate chapters add up.

**Task.** Count every remaining chapter and add its result to `total`, key by key. The `chapters` count adds with `+`, and so does your `Tally`.

In [ ]:
total = counter.count(chapters[0]["text"])
for chapter in chapters[1:]:
    output = counter.count(chapter["text"])
    total = {key: total[key] + output[key] for key in total}


print(total["chapters"], "chapters")
total["words"].total

The ten most common long words of the whole book:

In [ ]:
sorted(total["words"].counts.items(), key=lambda pair: pair[1], reverse=True)[:10]

## Bonus: Inheritance

Inheritance changes only what differs.

**Task.** Write `LongWords`, a `WordCounter` whose `min_length` is always 9. It defines only `__init__`, which calls `super().__init__(...)`.

In [ ]:
class LongWords(WordCounter):
    def __init__(self):
        super().__init__(min_length=9)


long_output = LongWords().count(chapters[0]["text"])
long_output["words"].total

**Done.** A class holds the settings, a method does the work, and results add with `+`: the same shape as your Higgs analysis, on a book.